In [ ]:
# Load libraries
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
import imagehash
from sklearn.metrics import roc_curve, auc, precision_score, recall_score, f1_score, precision_recall_curve
import matplotlib.pyplot as plt
from collections import defaultdict

# Load the FIND algorithm
from find.FINd_opt import FINDHasher
findHasher = FINDHasher()

## Subsample the data

In [ ]:
# Load image paths
image_dir = "/data/meme_images" # Update this path to your dataset location
all_paths = sorted([os.path.join(image_dir, f)
                    for f in os.listdir(image_dir)
                    if f.endswith(".jpg")])

# Get family label from filename (first 4 characters)
def get_label(path):
    return os.path.basename(path)[:4]

# Group paths by family
family_to_paths = defaultdict(list)
for p in all_paths:
    family_to_paths[get_label(p)].append(p)

# Compute number of images per family
family_sizes = np.array([len(v) for v in family_to_paths.values()])

print(f"Families: {len(family_sizes)}")
print(f"Images per family:")
print(f"  Min:    {family_sizes.min()}")
print(f"  Max:    {family_sizes.max()}")
print(f"  Mean:   {family_sizes.mean():.1f}")
print(f"  Median: {int(np.median(family_sizes))}")

In [ ]:
# Sample 10 images per family
random.seed(42)
k = 10
paths = []
for fam_paths in family_to_paths.values():
    paths.extend(random.sample(fam_paths, min(k, len(fam_paths))))

labels = np.array([get_label(p) for p in paths])

## Define core functions and algorithms

In [ ]:
# Compute hash using chosen algorithm
def compute_hash(img, hash_fn):
    h = hash_fn(img)
    return np.array(h.hash, dtype=np.uint8).ravel() # Convert to flat NumPy array (binary, consistent shape)

# Run FINd algorithm
def find_hash(img):
    return findHasher.fromImage(img)

# Run phash algorithm, setting phash size to 16 to get 256-bit hash (same length as FINd)
def phash(img):
    return imagehash.phash(img, hash_size=16)

# Available algorithms
algorithms = {
    "find": find_hash,
    "phash": phash
}

display_names = {
    "find": "FINd",
    "phash": "pHash",
}

# Compute Hamming distance
def hamming(a, b):
    return int(np.sum(a ^ b))

## Quick sanity check

Do similar images have the same label (i.e, come from the same family)?


In [ ]:
fn = find_hash  # change to test other algorithms

hashes = []

for p in paths:
    img = Image.open(p)
    hashes.append(compute_hash(img, fn))
    img.close()

hashes = np.array(hashes)

i = 0

dists = np.array([
    hamming(hashes[i], hashes[j]) if i != j else np.inf
    for j in range(len(hashes))
])

top5 = np.argsort(dists)[:5]

print("Query label:", labels[i])
print("Top 5 labels:", labels[top5])

## Classification

ROC/AUC, precision, recall, and F1

In [ ]:
def compute_accuracy(hashes, labels, num_pairs=500000):
    '''Compute accuracy metrics (ROC AUC, precision, recall, F1) by 
    sampling num_pairs of images'''
    n = len(hashes)
    hash_bits = hashes.shape[1]

    # Sample all pairs at once, resample any where i == j
    idx = np.random.randint(0, n, size=(num_pairs, 2)) # generate random image index pairs (i, j)
    collisions = idx[:, 0] == idx[:, 1] # check if two indices are the same (collision pairs)
    while collisions.any(): # resample collision pairs
        idx[collisions] = np.random.randint(0, n, size=(collisions.sum(), 2))
        collisions = idx[:, 0] == idx[:, 1]

    # Vectorized hamming distances (normalised to [0, 1]) and labels
    distances = np.sum(hashes[idx[:, 0]] ^ hashes[idx[:, 1]], axis=1)
    norm_distances = distances / hash_bits
    y_true = (labels[idx[:, 0]] == labels[idx[:, 1]]).astype(int)
    y_score = -norm_distances  # higher = more similar

    # Compute ROC curve and AUC
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)

    return fpr, tpr, roc_auc, y_true, norm_distances

In [ ]:
results = []
prf_curves = {}

all_labels = np.array([get_label(p) for p in all_paths])

plt.figure(figsize=(6, 6))

for name, fn in algorithms.items():
    label = display_names[name]
    print(f"\nRunning {label}...")

    all_hashes = []
    for p in all_paths:
        img = Image.open(p)
        all_hashes.append(compute_hash(img, fn))
        img.close()
    all_hashes = np.array(all_hashes)

    np.random.seed(42)
    fpr, tpr, roc_auc, y_true, norm_dist = compute_accuracy(all_hashes, all_labels)

    plt.plot(fpr, tpr, label=f"{label} (AUC={roc_auc:.3f})")

    # Precision/Recall/F1 across normalized thresholds
    pr, rc, pr_thr = precision_recall_curve(y_true, -norm_dist)
    norm_thr = -pr_thr
    f1_vals = 2 * pr[:-1] * rc[:-1] / (pr[:-1] + rc[:-1] + 1e-8)

    # Reorder thresholds from low to high for plotting
    sort_idx = np.argsort(norm_thr)
    norm_thr_sorted = norm_thr[sort_idx]
    pr_sorted = pr[:-1][sort_idx]
    rc_sorted = rc[:-1][sort_idx]
    f1_sorted = f1_vals[sort_idx]

    # Compute optimal threshold that maximizes F1 score
    opt_idx = np.argmax(f1_vals)
    T_opt = norm_thr[opt_idx]

    # Classify pairs as similar if distance <= optimal threshold
    y_pred = norm_dist <= T_opt

    results.append({
        "Algorithm": label,
        "ROC AUC": roc_auc,
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "Optimal threshold": T_opt,
    })

    prf_curves[label] = {
        "thresholds": norm_thr_sorted,
        "precision": pr_sorted,
        "recall": rc_sorted,
        "f1": f1_sorted,
        "opt_threshold": T_opt,
    }

# ROC curve
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.savefig("roc_curve.png", dpi=300, bbox_inches="tight")
plt.show()

# Results table at optimal threshold
df_results = pd.DataFrame(results).set_index("Algorithm").round(3)
display(df_results)

# Plot precision/Recall/F1 vs normalized threshold 
fig, axes = plt.subplots(len(prf_curves), 1, figsize=(6, 4 * len(prf_curves)), sharex=True)
if len(prf_curves) == 1:
    axes = [axes]

for ax, (label, data) in zip(axes, prf_curves.items()):
    ax.plot(data["thresholds"], data["precision"], label="Precision")
    ax.plot(data["thresholds"], data["recall"], label="Recall")
    ax.plot(data["thresholds"], data["f1"], label="F1")
    ax.axvline(data["opt_threshold"], color="black", linestyle="--", linewidth=1,
               label=f"Optimal ({data['opt_threshold']:.3f})")
    ax.set_ylabel("Score")
    ax.set_title(label)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend()

axes[-1].set_xlabel("Threshold")

fig.savefig("prf_threshold.png", dpi=300, bbox_inches="tight")
plt.show()

## Retrieval

Precision@k and MAP

In [ ]:
def retrieval_metrics(hashes, labels, ks):
    n = len(hashes)
    per_k = {k: [] for k in ks}
    avg_precisions = []

    for i in range(n):
        dists = np.sum(hashes[i] ^ hashes, axis=1).astype(float)
        dists[i] = np.inf

        sorted_idx = np.argsort(dists)
        relevant = (labels[sorted_idx] == labels[i])

        # Average precision for this query
        num_relevant = relevant.sum()
        if num_relevant > 0:
            hit_positions = np.where(relevant)[0]  # 0-indexed ranks of relevant items
            precisions = np.cumsum(relevant)[hit_positions] / (hit_positions + 1)
            avg_precisions.append(precisions.mean())

        for k in ks:
            per_k[k].append(relevant[:k].sum() / k)

    return {k: np.mean(v) for k, v in per_k.items()}, np.mean(avg_precisions)

In [ ]:
results_retrieval = {}

for name, fn in algorithms.items():
    label = display_names[name]
    print(f"\nRunning {label}...")

    hashes = []
    for p in paths:
        img = Image.open(p)
        hashes.append(compute_hash(img, fn))
        img.close()
    hashes = np.array(hashes)

    p_at_k, map_score = retrieval_metrics(hashes, labels, [1, 5, 9])
    results_retrieval[label] = {"P@1": p_at_k[1], "P@5": p_at_k[5], "P@9": p_at_k[9], "MAP": map_score}

df_retrieval = pd.DataFrame(results_retrieval).T.round(4)
display(df_retrieval)